In [ ]:
import os
from dotenv import find_dotenv, load_dotenv

# Always reload latest values from .env into the current notebook kernel.
ENV_PATH = find_dotenv(filename=".env", usecwd=True)
if not ENV_PATH:
    raise FileNotFoundError("Could not locate a .env file from the current working directory tree.")

load_dotenv(dotenv_path=ENV_PATH, override=True)

# OpenAI client setup

from pathlib import Path
from openai import OpenAI
import httpx

cert_env = os.environ.get("CERT_LLM", "").strip()
if not cert_env:
    raise EnvironmentError(
        "CERT_LLM is not set. Point it to your CA bundle file or directory."
    )

cert_path = Path(cert_env).expanduser()
# If CERT_LLM is a directory, use the default bundle filename inside it.
if cert_path.is_dir():
    cert_path = cert_path / "ca-bundle_lite.crt"

SSL_CERT = str(cert_path)
if not cert_path.exists():
    raise FileNotFoundError(f"Certificate bundle not found at: {SSL_CERT}")

BASE_URL = "https://litellm.icp.infineon.com"
MODEL = os.environ.get("OPENAI_MODEL", "gpt-5.2").strip()

http_client = httpx.Client(verify=SSL_CERT)
client = OpenAI(
    base_url=f"{BASE_URL}/v1",
    api_key=os.environ.get("OPENAI_API_KEY"),
    http_client=http_client,
)

model_list = client.models.list()
available_models = [m.id for m in model_list.data if getattr(m, "id", None)]
valid_models = [m for m in available_models if m != "no-default-models"]

if MODEL == "no-default-models":
    MODEL = ""

if MODEL and MODEL not in available_models:
    print(f"Requested model '{MODEL}' is not available for this key.")
    MODEL = valid_models[0] if valid_models else ""

if not MODEL:
    print("No valid model available from the proxy.")
    print("Set OPENAI_MODEL to a model your key can access and rerun this cell.")
    print(f"Models returned by /v1/models: {available_models}")
else:
    print(f"Using model: {MODEL}")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": "this is a test request, write a short poem",
            }
        ],
    )
    

from langchain.chat_models import init_chat_model

# Route through the LiteLLM proxy (reuses the SSL bundle + key); a bare model name
# would default to the public OpenAI endpoint, which the network refuses.
model = init_chat_model(
    MODEL,
    model_provider="openai",
    base_url=f"{BASE_URL}/v1",
    api_key=os.environ.get("OPENAI_API_KEY"),
    http_client=http_client,
)

Using model: gpt-5.2


# Streaming and Batch 

#### Streaming the model output

In [3]:
model.stream("Write a 200 words story about a lion and a mouse.")

<generator object BaseChatModel.stream at 0x0000015C02EA1970>

In [5]:
for chunk in model.stream("Write a 2000 words story about a lion and a mouse."):
    print(chunk.text, end="")

The sun rose over the savanna like a slow-burning ember, turning the tall grass into a shimmering sea of gold. Morning wind slid between acacia trees and carried the scent of dust, dry earth, and distant water. In the cool hour before the heat thickened, the world felt wide and quiet—so quiet that even the smallest sounds seemed important: the click of beetle legs against a stone, the hiss of grass parting around a lizard, the soft, steady breathing of a great lion asleep in the shade.

His name, among the lions, was **Kavumo**—a name that meant *the one who stands like a hill*. He was not the oldest lion on the plains, nor the largest, but he carried himself with a calm weight that made other animals watch their steps. His mane, dark as storm clouds at the roots, spread over his shoulders like a cloak. His paws, wide and scarred, had walked through seasons of hunger and plenty. He was a hunter, a protector, and sometimes—when the day was long and the pride was fed—he was simply a crea

### Batch
Batching a collection of independent requests to a model can significantly improve performance and reduce cost as the processing can be done in parallel

In [10]:
responses = model.batch([
    "what is the capital of France?",
    "why is the sky blue?",
    "how do airplanes fly?",
    "when was the first moon landing?"],
    config={
        "max_concurrency": 5,
    }
)

for response in responses:
    print(response.content)

Paris.
Sunlight looks white, but it’s made of many colors. When it enters Earth’s atmosphere, it hits tiny molecules of nitrogen and oxygen that scatter the light in all directions. This scattering is much stronger for shorter wavelengths (blue and violet) than for longer wavelengths (red). As a result, blue light gets scattered across the sky more, so from most directions you look, you’re seeing scattered blue light—making the sky appear blue.

A couple related details:
- Violet scatters even more than blue, but our eyes are less sensitive to violet, and some violet gets absorbed higher in the atmosphere, so the sky looks predominantly blue.
- At sunrise and sunset, sunlight passes through much more atmosphere, so the blue light gets scattered out of the direct path and the remaining light looks red/orange.
Airplanes fly because their wings generate an upward force called **lift** that can balance (and exceed) the airplane’s weight.

### 1) Lift: the wing pushes air down
A wing moving